In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('adi-dev')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/20 21:34:03 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/20 21:34:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 21:34:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import functions as F

from adi.io import CsvStore, TrialBalanceRepository
from adi.pipeline import TrialBalancePipeline
from adi.config.settings import TABLE_PATHS
from adi.enrichments import (
    TransformationManager, 
    ReferenceManager
)

from finmap import FinMapClient

In [3]:
store = CsvStore(spark, table_paths=TABLE_PATHS)
repository = TrialBalanceRepository(store)

transformation_manager = TransformationManager(spark)
reference_manager = ReferenceManager(spark)

finmap = FinMapClient.from_csv(
    spark=spark,
    metadata_path='data/reference/mapping_meta.csv',
    data_path='data/reference/mapping_data.csv',
)

pipeline = TrialBalancePipeline(
    transformation_manager=transformation_manager,
    reference_manager=reference_manager,
    finmap=finmap,
)

In [4]:
df_source = repository.read_source()

df_source.show()

+--------+----------+----------+-----------+----------+----------+-------------+-------------+-------------------+--------------+-----------------+-------------+-------------+--------------------+------------------+---------------------+----------------------+-----------+
|BATCH_ID|EXTRACT_DT|  AS_OF_DT|BUSINESS_DT|SRC_APP_CD|SRC_APP_NM|SRC_RECORD_ID|SRC_ENTITY_CD|SRC_BOOKING_DEPT_CD|SRC_ACCOUNT_ID|   SRC_ACCOUNT_NM|SRC_CLIENT_ID|SRC_CLIENT_NM|      SRC_MEASURE_NM|SRC_MEASURE_CCY_CD|SRC_MEASURE_TRANS_AMT|POSTING_MEASURE_CCY_CD|CPTY_REF_ID|
+--------+----------+----------+-----------+----------+----------+-------------+-------------+-------------------+--------------+-----------------+-------------+-------------+--------------------+------------------+---------------------+----------------------+-----------+
|       1|2025-03-31|2025-03-31| 2025-03-31|     11392|    IMPACT|        rec-1|          NKC|               4613|        100011|  NKC TD CLEARING|       308282|         NULL|src_pr

In [5]:
df_source = repository.read_source()

df_staging = pipeline.staging(df_source)
repository.write_staging(df_staging)

df_staging_reloaded = repository.read_staging()

display(df_staging_reloaded.toPandas())

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,...,COA_RULE_ID,ENTITY_SUN_ID,CLIENT_ID_TYPE,INTERGROUP_IND,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT
0,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,None,243120,THIRDPARTY,TP,previous_day_balance,REPORTABLE,CAD,3424081.950000000000,1.000000000000,3424081.950000000000
1,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,None,243120,THIRDPARTY,TP,current_day_debit_balance,REPORTABLE,CAD,0E-12,1.000000000000,0E-12
2,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,None,243120,THIRDPARTY,TP,current_day_credit_balance,REPORTABLE,CAD,-709.880000000000,1.000000000000,-709.880000000000
3,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,None,243120,THIRDPARTY,TP,current_day_eod_balance,REPORTABLE,CAD,3423372.070000000000,1.000000000000,3423372.070000000000
4,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,None,243120,THIRDPARTY,TP,back_value_adjusted_balance,REPORTABLE,CAD,0E-12,1.000000000000,0E-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,None,100774,None,None,current_day_debit_balance,REPORTABLE,USD,0E-12,1.000000000000,0E-12
92,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,None,100774,None,None,current_day_credit_balance,REPORTABLE,USD,0E-12,1.000000000000,0E-12
93,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,None,100774,None,None,current_day_eod_balance,REPORTABLE,USD,-27108.500000000000,1.000000000000,-27108.500000000000
94,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,None,100774,None,None,back_value_adjusted_balance,REPORTABLE,USD,0E-12,1.000000000000,0E-12


In [6]:
(
    df_staging_reloaded
    .select('SRC_RECORD_ID', 'SRC_CLIENT_ID', 'CPTY_REF_ID', 'CLIENT_ID_TYPE', 'INTERGROUP_IND')
    .filter(F.col('SRC_RECORD_ID') == 'rec-16')
    .show()
)

+-------------+-------------+-----------+--------------+--------------+
|SRC_RECORD_ID|SRC_CLIENT_ID|CPTY_REF_ID|CLIENT_ID_TYPE|INTERGROUP_IND|
+-------------+-------------+-----------+--------------+--------------+
|       rec-16|         NULL|       NULL|          NULL|          NULL|
|       rec-16|         NULL|       NULL|          NULL|          NULL|
|       rec-16|         NULL|       NULL|          NULL|          NULL|
|       rec-16|         NULL|       NULL|          NULL|          NULL|
|       rec-16|         NULL|       NULL|          NULL|          NULL|
|       rec-16|         NULL|       NULL|          NULL|          NULL|
+-------------+-------------+-----------+--------------+--------------+

